# 1 Load Data

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../Data/student_performance_data.csv")
df.drop(columns=['overall_score','student_id'], inplace=True)
df

,gender,study_hours_per_day,attendance_percentage,assignment_score,midterm_score,final_exam_score,participation_score,internet_access,extra_classes,parent_education,sleep_hours,grade
0,Male,4.54,69.98,36.47,70.70,53.10,17.96,Yes,No,Master,8.09,D
1,Female,5.26,84.80,34.25,27.92,87.17,11.29,No,Yes,Bachelor,4.73,D
2,Male,8.69,73.76,72.29,70.92,99.61,76.10,No,Yes,PhD,8.73,B
3,Male,4.06,45.00,97.63,31.73,88.85,33.55,No,No,Bachelor,8.22,C
4,Male,8.83,51.13,65.19,78.28,54.23,88.99,No,No,Bachelor,8.59,C
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Female,6.86,59.65,60.41,39.51,66.43,44.97,Yes,Yes,High School,8.56,C
9996,Male,2.60,83.62,62.45,48.96,81.40,45.11,Yes,Yes,Bachelor,4.21,C
9997,Female,1.46,95.40,67.08,51.51,87.58,65.49,Yes,No,Bachelor,4.72,B
9998,Female,7.15,78.24,97.73,46.51,75.49,61.21,No,No,Bachelor,5.28,B


In [2]:
df['grade'].value_counts()

grade
C    5073
B    2704
D    2008
A     154
F      61
Name: count, dtype: int64

In [3]:
df = df[df['grade'].isin(['C','B'])]

In [4]:
df.columns

Index(['gender', 'study_hours_per_day', 'attendance_percentage',
       'assignment_score', 'midterm_score', 'final_exam_score',
       'participation_score', 'internet_access', 'extra_classes',
       'parent_education', 'sleep_hours', 'grade'],
      dtype='str')

##### ohe -> 'gender'
##### ordinal -> 'parent_education'
##### Label -> 'internet_access', 'extra_classes'
##### scaling_transform -> 'study_hours_per_day', 'attendance_percentage','assignment_score', 'participation_score',midterm_score', 'final_exam_score', 'sleep_hours'



In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler, PowerTransformer

scaling_transform = Pipeline([
    ("scl", StandardScaler()),
    ('power', PowerTransformer())
])

preprocessor = ColumnTransformer(transformers=[
    ("ohe", OneHotEncoder(sparse_output=False,handle_unknown='ignore'),['gender']),
    ("ord", OrdinalEncoder(categories=[['High School','Bachelor','Master','PhD']]), ['parent_education']),
    ("ord_bin", OrdinalEncoder(), ['internet_access', 'extra_classes']),
    ("scaling_transform", scaling_transform, ['study_hours_per_day', 'attendance_percentage','assignment_score', 'midterm_score', 'final_exam_score','participation_score', 'sleep_hours'])
    
],remainder='passthrough')

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:-1], df.iloc[:,-1], test_size=0.2, random_state=42)

In [7]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [8]:
lb = LabelEncoder()
y_train = lb.fit_transform(y_train)
y_test = lb.transform(y_test)

In [9]:
X_train

array([[ 1.        ,  0.        ,  2.        , ...,  1.55723615,
        -0.75892707, -0.89485911],
       [ 0.        ,  1.        ,  0.        , ...,  0.72436296,
        -0.29935487,  0.29754975],
       [ 1.        ,  0.        ,  0.        , ...,  1.39260359,
         1.18104473, -0.11537298],
       ...,
       [ 0.        ,  1.        ,  1.        , ..., -1.84484122,
         1.65800829,  0.10459476],
       [ 0.        ,  1.        ,  0.        , ..., -0.78651245,
        -0.41800821, -1.37117388],
       [ 1.        ,  0.        ,  2.        , ...,  1.00338957,
         1.44685614, -1.19442421]], shape=(6221, 12))

In [10]:
import torch
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(np.asarray(y_train).astype(np.float32))   # Series - > Dataframe
y_test = torch.from_numpy(np.asarray(y_test).astype(np.float32))

# 3. Build Model

In [11]:
import torch.nn as nn
class MySimplePerceptron(nn.Module):
    def __init__(self, X_train):
        super().__init__()
        self.linear1 = nn.Linear(X_train.shape[1], 6)
        self.relu1 = nn.ReLU()
        
        self.linear2 = nn.Linear(6,3)
        self.relu2 = nn.ReLU()
        
        self.linear3 = nn.Linear(3, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, X_train):
        out = self.linear1(X_train)
        out = self.relu1(out)
        
        out = self.linear2(out)
        out = self.relu2(out)
        
        out = self.linear3(out)
        out = self.sigmoid(out)
        return out
    
    def loss_function(self, y_pred, y_train):
        epsilon = 1e-8
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
        loss = (  -y_train*torch.log(y_pred) - (1-y_train)*torch.log(1-y_pred) ).mean()
        return loss

# 4. Train Model

In [12]:
def train_model(learning_rate = 0.1,epochs = 100):
    model = MySimplePerceptron(X_train)
    
    for epoch in range(epochs):
        # forward propagation
        y_pred = model.forward(X_train)
        
        # loss calculate
        loss = model.loss_function(y_pred, y_train)
        
        # backpropagation
        loss.backward()
        
        # update weight and bias
        with torch.no_grad():
            model.linear1.weight -= (learning_rate * model.linear1.weight.grad)
            model.linear1.bias -= (learning_rate * model.linear1.bias.grad)
            
            model.linear2.weight -= (learning_rate * model.linear2.weight.grad)
            model.linear2.bias -= (learning_rate * model.linear2.bias.grad)
            
            model.linear3.weight -= (learning_rate * model.linear3.weight.grad)
            model.linear3.bias -= (learning_rate * model.linear3.bias.grad)
            
        # reinitialize gradient
        model.linear1.weight.grad.zero_()
        model.linear1.bias.grad.zero_()
        model.linear2.weight.grad.zero_()
        model.linear2.bias.grad.zero_()
        model.linear3.weight.grad.zero_()
        model.linear3.bias.grad.zero_()
        
        print(f"Epoch: {epoch+1}, Loss: {loss.item()}")
        
    return model

In [13]:
model = train_model(epochs=40, learning_rate=0.01)

Epoch: 1, Loss: 0.6865712404251099
Epoch: 2, Loss: 0.6863452196121216
Epoch: 3, Loss: 0.6861203908920288
Epoch: 4, Loss: 0.6858968734741211
Epoch: 5, Loss: 0.6856746077537537
Epoch: 6, Loss: 0.685453474521637
Epoch: 7, Loss: 0.6852335929870605
Epoch: 8, Loss: 0.6850149035453796
Epoch: 9, Loss: 0.6847973465919495
Epoch: 10, Loss: 0.6845810413360596
Epoch: 11, Loss: 0.6843658089637756
Epoch: 12, Loss: 0.6841518878936768
Epoch: 13, Loss: 0.6839389801025391
Epoch: 14, Loss: 0.6837272047996521
Epoch: 15, Loss: 0.6835166215896606
Epoch: 16, Loss: 0.6833072304725647
Epoch: 17, Loss: 0.6830988526344299
Epoch: 18, Loss: 0.6828917264938354
Epoch: 19, Loss: 0.6826856732368469
Epoch: 20, Loss: 0.6824808716773987
Epoch: 21, Loss: 0.6822770833969116
Epoch: 22, Loss: 0.6820743680000305
Epoch: 23, Loss: 0.6818729639053345
Epoch: 24, Loss: 0.6816725134849548
Epoch: 25, Loss: 0.6814732551574707
Epoch: 26, Loss: 0.6812750101089478
Epoch: 27, Loss: 0.6810779571533203
Epoch: 28, Loss: 0.680881917476654
Epo

# 5. Evaluate Model

In [14]:
def evaluate_model(threshold=0.5):
    with torch.no_grad():  
        y_pred = model.forward(X_test)
        y_pred = (y_pred > threshold).float()
        
        accuracy = (y_test == y_pred).float().mean()
        print(f"Accuracy: {accuracy}")

In [15]:
evaluate_model(threshold=0.5)

Accuracy: 0.6696658134460449
